# Khám phá dữ liệu (EDA) - Online Retail II

Notebook này nhằm mục đích hiểu rõ đặc điểm thực tế của dữ liệu Online Retail II và phát hiện các vấn đề chất lượng đặc thù của dataset.

In [1]:
import pandas as pd
import numpy as np

# Đọc dữ liệu
df = pd.read_csv('../data/raw/online_retail_II.csv', encoding='latin1')
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


## 1. Khảo sát tổng quan
Kiểm tra shape, kiểu dữ liệu, tỷ lệ missing từng cột, số lượng duplicate.

In [2]:
print("Shape:", df.shape)
print("\n--- Info ---")
df.info()

Shape: (541910, 8)

--- Info ---
<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      541910 non-null  str    
 1   StockCode    541910 non-null  str    
 2   Description  540456 non-null  str    
 3   Quantity     541910 non-null  int64  
 4   InvoiceDate  541910 non-null  str    
 5   Price        541910 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      541910 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 66.2 MB


In [3]:
print("\n--- Describe ---")
df.describe()


--- Describe ---


,Quantity,Price,Customer ID
count,541910.000000,541910.000000,406830.000000
mean,9.552234,4.611138,15287.684160
std,218.080957,96.759765,1713.603074
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [4]:
print("\n--- Missing Values ---")
print(df.isnull().sum())
print("\nTỷ lệ missing CustomerID:", df['Customer ID'].isnull().mean())


--- Missing Values ---
Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

Tỷ lệ missing CustomerID: 0.2492664833643963


In [5]:
print("\n--- Duplicates ---")
print("Số dòng duplicate:", df.duplicated().sum())


--- Duplicates ---
Số dòng duplicate: 5268


## 2. Định lượng các đặc thù riêng
Tỷ lệ hóa đơn hủy, Quantity âm, StockCode đặc biệt, Price = 0.

In [14]:
# Tỷ lệ hóa đơn hủy
df['IsCancelled'] = df['Invoice'].astype(str).str.startswith('C')
print("Tỷ lệ hóa đơn hủy:", df['IsCancelled'].value_counts())

Tỷ lệ hóa đơn hủy: IsCancelled
False    532622
True       9288
Name: count, dtype: int64


In [7]:
# Tương quan hóa đơn hủy và Quantity âm
print(df.groupby('IsCancelled')['Quantity'].describe())

                count       mean          std      min  25%  50%   75%  \
IsCancelled                                                              
False        532622.0  10.239954   159.593402  -9600.0  1.0  3.0  10.0   
True           9288.0 -29.885228  1145.786965 -80995.0 -6.0 -2.0  -1.0   

                 max  
IsCancelled           
False        80995.0  
True            -1.0  


In [8]:
# Các StockCode đặc biệt
special_codes = ['POST', 'DOT', 'M', 'BANK CHARGES', 'C2', 'ADJUST', 'CRUK']
special_count = df[df['StockCode'].astype(str).isin(special_codes)].shape[0]
print("Số dòng có StockCode đặc biệt (POST, DOT, ...):", special_count)

Số dòng có StockCode đặc biệt (POST, DOT, ...): 2735


In [9]:
# Price = 0 hoặc âm
print("Số dòng có Price == 0:", (df['Price'] == 0).sum())
print("Số dòng có Price < 0:", (df['Price'] < 0).sum())

Số dòng có Price == 0: 2515
Số dòng có Price < 0: 2


## 3. Chạy thử Module Làm sạch và kiểm tra

In [10]:
import sys
sys.path.append('..')
from src.data.cleaning import clean_pipeline

# Chạy pipeline làm sạch (sẽ in log ra console/output)
df_clean = clean_pipeline(df)

print("\n--- Kiểm tra lại kết quả ---")
print("Shape sau khi clean:", df_clean.shape)
df_clean.head()

--- Starting cleaning pipeline. Initial shape: (541910, 9) ---
[remove_duplicates] Removed 5268 duplicate rows.
[fix_dtypes] Fixing data types...


c:\Users\NHAN\OneDrive\AIO_Conquer_Module_2\customer_segmentation\notebooks\..\src\data\cleaning.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[invoice_date_col] = pd.to_datetime(df[invoice_date_col], errors='coerce')


[flag_invalid_date] Flagged 0 rows with invalid InvoiceDate.
[flag_cancelled] Flagged 9251 cancelled invoices.
[flag_missing_customer] Flagged 135037 rows missing Customer ID.
[flag_special_stockcode] Flagged 2907 rows with special StockCodes.
[flag_price_anomaly] Flagged 2512 rows with Price <= 0.
--- Finished cleaning pipeline. Final shape: (536642, 14) ---

--- Kiểm tra lại kết quả ---
Shape sau khi clean: (536642, 14)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled,HasInvalidDate,HasCustomerID,IsServiceCode,PriceAnomaly,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,False,False,True,False,False,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,True,False,False,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,False,False,True,False,False,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,True,False,False,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,True,False,False,20.34


In [11]:
# Đối chiếu tay vài dòng hóa đơn hủy
df_clean[df_clean['IsCancelled'] == True].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled,HasInvalidDate,HasCustomerID,IsServiceCode,PriceAnomaly,TotalPrice
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527,United Kingdom,True,False,True,True,False,-27.50
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311,United Kingdom,True,False,True,False,False,-4.65
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548,United Kingdom,True,False,True,False,False,-19.80
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548,United Kingdom,True,False,True,False,False,-6.96
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548,United Kingdom,True,False,True,False,False,-6.96
